In [2]:
from pgmpy.models import DiscreteBayesianNetwork
from pgmpy.factors.discrete import TabularCPD, State
from pgmpy.inference import VariableElimination
from pgmpy.sampling import BayesianModelSampling

In [12]:
model = DiscreteBayesianNetwork([
    ("heatwave", "overload"),
    ("heatwave", "powercut"),
    ("overload", "powercut"),
    ("powercut", "cancel"),
])

cpd_h = TabularCPD("heatwave", 2, [[0.3], [0.7]],
                   state_names={"heatwave": ["yes", "no"]})

cpd_a = TabularCPD("overload", 2,
                   [[0.6, 0.1],
                    [0.4, 0.9]],
                   evidence=["heatwave"], evidence_card=[2],
                   state_names={"overload": ["yes", "no"], "heatwave": ["yes", "no"]})

cpd_c = TabularCPD("powercut", 2,
                   [[0.7, 0.2, 0.5, 0.05],
                    [0.3, 0.8, 0.5, 0.95]],
                   evidence=["heatwave", "overload"], evidence_card=[2, 2],
                   state_names={"powercut": ["yes", "no"],
                                "heatwave": ["yes", "no"],
                                "overload": ["yes", "no"]})

cpd_l = TabularCPD("cancel", 2,
                   [[0.8, 0.1],
                    [0.2, 0.9]],
                   evidence=["powercut"], evidence_card=[2],
                   state_names={"cancel": ["yes", "no"], "powercut": ["yes", "no"]})

model.add_cpds(cpd_h, cpd_a, cpd_c, cpd_l)
print(model.check_model())

True


In [13]:
infer = VariableElimination(model)
result = infer.query(variables=["heatwave"], evidence={"cancel": "yes"})
print(result)

+---------------+-----------------+
| heatwave      |   phi(heatwave) |
+===============+=================+
| heatwave(yes) |          0.5367 |
+---------------+-----------------+
| heatwave(no)  |          0.4633 |
+---------------+-----------------+


In [15]:
sampler = BayesianModelSampling(model)
samples = sampler.rejection_sample(evidence=[State("cancel", "yes")], size=10000)
estimate = (samples["heatwave"] == "yes").mean()
print("Estimated P(heatwave = yes | cancel = yes) =", round(estimate, 4))

100%|██████████| 10000/10000 [00:00<00:00, 68794.27it/s]

Estimated P(heatwave = yes | cancel = yes) = 0.5367


Variable elimination gives the exact answer P(heatwave = yes | cancel = yes) = 0.5367, the same as the hand calculation in Exercise 4. Rejection sampling with 10,000 samples gave an estimate of 0.5367, which matches the exact answer to four decimal places here; sampling is random, so the estimate can change slightly on each run, but with many samples it stays very close to the exact value.
